
# ♟️ Play Against Stockfish — Colab Edition

This notebook gives you an interactive chess board and lets you play against **Stockfish**.

### What makes it winnable?
Full-strength Stockfish is far beyond human world-champion strength. This notebook includes a **Win-friendly mode** that still asks Stockfish to analyze the position, but intentionally chooses from weaker Stockfish candidate moves.

You can control:

- **Your side:** White or Black
- **Stockfish skill:** 0–20
- **Thinking time**
- **Win-friendly mode**
- **Blunder chance**
- **Hints**
- **Undo**
- **Reset game**

### How to play
1. Run the install cell.
2. Run the game cell.
3. Enter moves in **SAN** (`e4`, `Nf3`, `O-O`, `Qxd5+`) or **UCI** (`e2e4`, `g1f3`).
4. Click **Play move**.

> Start with Skill 0, Win-friendly mode ON, and Blunder chance around 50–70%.


In [ ]:

# Install Stockfish + Python chess tools
!apt-get -qq update
!apt-get -qq install -y stockfish
!pip -q install python-chess ipywidgets

print("✅ Installed Stockfish and python-chess.")


In [ ]:

import chess
import chess.engine
import chess.svg
import random
import shutil
import os

import ipywidgets as widgets
from IPython.display import display, SVG, clear_output

# -----------------------------
# Find Stockfish
# -----------------------------
candidate_paths = [
    shutil.which("stockfish"),
    "/usr/games/stockfish",
    "/usr/bin/stockfish",
]
STOCKFISH_PATH = next((p for p in candidate_paths if p and os.path.exists(p)), None)

if STOCKFISH_PATH is None:
    raise FileNotFoundError("Stockfish was not found. Re-run the install cell above.")

engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
print(f"✅ Stockfish running from: {STOCKFISH_PATH}")

# -----------------------------
# Game state
# -----------------------------
board = chess.Board()
last_move = None

# -----------------------------
# Widgets
# -----------------------------
side_dropdown = widgets.Dropdown(
    options=[("White", chess.WHITE), ("Black", chess.BLACK)],
    value=chess.WHITE,
    description="You play:"
)

skill_slider = widgets.IntSlider(
    value=0, min=0, max=20, step=1, description="Skill:"
)

think_slider = widgets.FloatSlider(
    value=0.10, min=0.03, max=2.0, step=0.03,
    readout_format=".2f", description="Think sec:"
)

friendly_checkbox = widgets.Checkbox(
    value=True, description="Win-friendly mode"
)

blunder_slider = widgets.IntSlider(
    value=60, min=0, max=100, step=5, description="Blunder %:"
)

move_box = widgets.Text(
    placeholder="e4, Nf3, O-O, e2e4 ...",
    description="Your move:"
)

play_button = widgets.Button(description="Play move", button_style="success")
hint_button = widgets.Button(description="Hint", button_style="info")
undo_button = widgets.Button(description="Undo", button_style="warning")
reset_button = widgets.Button(description="Reset")

board_output = widgets.Output()
status_output = widgets.Output()

# -----------------------------
# Engine helpers
# -----------------------------
def configure_engine():
    try:
        engine.configure({"Skill Level": int(skill_slider.value)})
    except Exception:
        pass

def user_color():
    return side_dropdown.value

def render_board(message=None):
    with board_output:
        clear_output(wait=True)
        svg = chess.svg.board(
            board=board,
            orientation=user_color(),
            lastmove=last_move,
            size=520
        )
        display(SVG(svg))

        if board.is_game_over():
            outcome = board.outcome()
            print(f"\n🏁 Game over: {board.result()} — {outcome}")
        elif message:
            print(f"\n{message}")
        else:
            turn = "White" if board.turn == chess.WHITE else "Black"
            print(f"\n{turn} to move")

def parse_user_move(text):
    text = text.strip()
    if not text:
        raise ValueError("Enter a move first.")

    try:
        return board.parse_san(text)
    except Exception:
        pass

    try:
        move = chess.Move.from_uci(text.lower())
        if move in board.legal_moves:
            return move
    except Exception:
        pass

    raise ValueError(f"'{text}' is not a legal SAN or UCI move here.")

def stockfish_candidates(max_candidates=8):
    configure_engine()

    legal_count = board.legal_moves.count()
    if legal_count == 0:
        return []

    k = min(max_candidates, legal_count)
    infos = engine.analyse(
        board,
        chess.engine.Limit(time=float(think_slider.value)),
        multipv=max(1, k)
    )

    if isinstance(infos, dict):
        infos = [infos]

    candidates = []
    for info in infos:
        pv = info.get("pv", [])
        if pv:
            move = pv[0]
            if move in board.legal_moves:
                candidates.append((move, info))

    return candidates

def choose_stockfish_move():
    # Normal mode uses Stockfish's top move.
    # Win-friendly mode occasionally selects a weaker Stockfish MultiPV move.
    candidates = stockfish_candidates(max_candidates=8)

    if not candidates:
        return None

    if not friendly_checkbox.value:
        return candidates[0][0]

    blunder_probability = blunder_slider.value / 100.0

    if len(candidates) == 1 or random.random() > blunder_probability:
        return candidates[0][0]

    start = max(1, len(candidates) // 2)
    weaker = candidates[start:] or candidates[1:] or candidates

    # Bias toward the weaker end of the candidate list.
    weights = list(range(1, len(weaker) + 1))
    return random.choices(
        [m for m, _ in weaker],
        weights=weights,
        k=1
    )[0]

def engine_turn():
    global last_move

    if board.is_game_over() or board.turn == user_color():
        return

    with status_output:
        clear_output(wait=True)
        print("🤖 Stockfish is thinking...")

    move = choose_stockfish_move()

    if move is None:
        render_board()
        return

    san = board.san(move)
    board.push(move)
    last_move = move

    with status_output:
        clear_output(wait=True)
        mode = "handicapped Stockfish" if friendly_checkbox.value else "Stockfish"
        print(f"🤖 {mode} played: {san} ({move.uci()})")

    render_board()

def maybe_engine_first():
    if not board.is_game_over() and board.turn != user_color():
        engine_turn()

# -----------------------------
# Controls
# -----------------------------
def on_play_clicked(_):
    global last_move

    if board.is_game_over():
        render_board("The game is already over. Click Reset.")
        return

    if board.turn != user_color():
        render_board("It is Stockfish's turn.")
        return

    try:
        move = parse_user_move(move_box.value)
        san = board.san(move)
        board.push(move)
        last_move = move
        move_box.value = ""

        with status_output:
            clear_output(wait=True)
            print(f"🧑 You played: {san} ({move.uci()})")

        render_board()

        if not board.is_game_over():
            engine_turn()

    except Exception as exc:
        with status_output:
            clear_output(wait=True)
            print(f"❌ {exc}")

def on_hint_clicked(_):
    if board.is_game_over():
        return

    if board.turn != user_color():
        with status_output:
            clear_output(wait=True)
            print("Hint is available on your turn.")
        return

    configure_engine()
    info = engine.analyse(
        board,
        chess.engine.Limit(time=max(0.15, float(think_slider.value)))
    )
    pv = info.get("pv", [])
    if pv:
        best = pv[0]
        with status_output:
            clear_output(wait=True)
            print(f"💡 Hint: {board.san(best)} ({best.uci()})")

def on_undo_clicked(_):
    global last_move

    pops = min(2, len(board.move_stack))
    for _ in range(pops):
        board.pop()

    last_move = board.peek() if board.move_stack else None

    with status_output:
        clear_output(wait=True)
        print("↩️ Took back the last move pair.")

    render_board()
    maybe_engine_first()

def on_reset_clicked(_):
    global board, last_move
    board = chess.Board()
    last_move = None
    move_box.value = ""

    with status_output:
        clear_output(wait=True)
        print("🔄 New game started.")

    render_board()
    maybe_engine_first()

def on_side_change(change):
    if change["name"] == "value":
        on_reset_clicked(None)

play_button.on_click(on_play_clicked)
hint_button.on_click(on_hint_clicked)
undo_button.on_click(on_undo_clicked)
reset_button.on_click(on_reset_clicked)
side_dropdown.observe(on_side_change, names="value")

# -----------------------------
# Layout
# -----------------------------
controls = widgets.VBox([
    widgets.HTML("<h3>⚙️ Game settings</h3>"),
    side_dropdown,
    skill_slider,
    think_slider,
    friendly_checkbox,
    blunder_slider,
    widgets.HTML("<hr><b>Enter SAN or UCI:</b>"),
    move_box,
    widgets.HBox([play_button, hint_button]),
    widgets.HBox([undo_button, reset_button]),
])

game_ui = widgets.HBox([
    controls,
    widgets.VBox([board_output, status_output])
])

display(game_ui)
render_board("Ready. Make your move!")
maybe_engine_first()



## Recommended settings if your goal is to win

| Level | Skill | Blunder chance | Thinking time |
|---|---:|---:|---:|
| Beginner-friendly | 0 | 70% | 0.05 s |
| Easy | 2 | 50% | 0.10 s |
| Medium | 5 | 30% | 0.20 s |
| Hard | 10 | 10% | 0.50 s |
| Very hard | 15 | 0% | 1.00 s |
| Full-ish Stockfish | 20 | OFF | 2.00 s |

**Important:** beating unrestricted full-strength Stockfish from the normal starting position is not a realistic human target. The win-friendly mode is deliberately handicapped so you can practice, win, and then raise the difficulty progressively.


In [ ]:

# Optional: run when you're done.
try:
    engine.quit()
    print("✅ Stockfish shut down.")
except Exception:
    print("Stockfish was already stopped.")
